In [ ]:
# Merge group myeloid objects and re-integrate with scVI.
import os, glob
import scanpy as sc
import anndata as ad
import pandas as pd

input_dir = "/home/yanjun3/atlas/heart_sn"

# read and concat (new API), keep original barcodes
files = sorted(glob.glob(os.path.join(input_dir, "group*_myeloid.h5ad")))
adatas = [sc.read(f) for f in files]
adata = ad.concat(adatas, join="outer", label="group", keys=[os.path.basename(f) for f in files], index_unique=None)

# fix write error: drop per-batch HVG flags that become mixed-type after concat
adata.var = adata.var.drop(columns=[c for c in adata.var.columns if str(c).startswith("highly_variable")], errors="ignore")

# ensure dtypes are write-safe
bool_cols = adata.var.select_dtypes(include=["boolean"]).columns
adata.var[bool_cols] = adata.var[bool_cols].fillna(False).astype(bool)
obj_cols = adata.var.select_dtypes(include=["object"]).columns
adata.var[obj_cols] = adata.var[obj_cols].astype(str)

# write
out_path = os.path.join(input_dir, "heart_myeloid_merged.h5ad")
adata.write(out_path, compression="gzip")


In [ ]:
del adatas

In [ ]:
input_dir = "/home/yanjun3/atlas/heart_sn"

In [ ]:
adata.obs_names_make_unique()

In [ ]:
adata

In [ ]:
import os, glob
import scanpy as sc
import anndata as ad
import pandas as pd

import scvi
import seaborn as sns
import torch
from rich import print
from scib_metrics.benchmark import Benchmarker

In [ ]:
sc.set_figure_params(figsize=(6, 6),  facecolor="white",
    frameon=True,   dpi = 300, format = 'png')
sc.settings.figdir = "/home/yanjun3/atlas/heart_sn/figures"
sns.set_theme()
torch.set_float32_matmul_precision("high")

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

In [ ]:
batch_key = "sample"

In [ ]:
sc.pp.highly_variable_genes(adata)
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
adata_hvg 

In [ ]:
# scVI integration
scvi.model.SCVI.setup_anndata(adata_hvg, layer="counts", batch_key=batch_key)
adata_hvg

In [ ]:
model_scvi = scvi.model.SCVI(adata_hvg)
model_scvi

In [ ]:
model_scvi.view_anndata_setup()

In [ ]:
model_scvi = scvi.model.SCVI(adata_hvg)
model_scvi

In [ ]:
import numpy as np
max_epochs_scvi = np.min([round((20000 / adata.n_obs) * 400), 400])
max_epochs_scvi

In [ ]:
model_scvi.train()

In [ ]:
adata.obsm["X_scVI"] = model_scvi.get_latent_representation()

In [ ]:
adata_hvg.obsm["X_scVI"] = model_scvi.get_latent_representation()

In [ ]:
# Save result
adata.write('myeloid_scvi_v1.h5ad')

In [ ]:
# Load data
adata = sc.read('myeloid_scvi_v1.h5ad')

In [ ]:
adata_hvg.write('myeloid_hvg_scvi.h5ad')

In [ ]:
# Neighbours, UMAP and clustering
sc.pp.neighbors(adata, use_rep="X_scVI")
sc.tl.umap(adata)

In [ ]:
sc.tl.leiden(adata, key_added="leiden_res0_5", resolution=0.5)
sc.pl.umap(
    adata,
    color=["leiden_res0_5"]
    ,
    frameon=False,
    ncols=1, size = 30, legend_loc = 'on data',
    save = '0_5'
)

In [ ]:
sc.tl.leiden(adata, key_added="leiden_res0_3", resolution=0.3)
sc.pl.umap(
    adata,
    color=["leiden_res0_3"]
    ,
    frameon=False,
    ncols=1, size = 30, legend_loc = 'on data',
    save = '0_3_v1.png'
)

In [ ]:
sc.tl.leiden(adata, key_added="leiden_res0_3", resolution=0.3)
sc.pl.umap(
    adata,
    color=["leiden_res0_3"]
    ,
    frameon=False,
    ncols=1, size = 30, legend_loc = 'on data',
    save = '0_3'
)

In [ ]:
# Differential expression
sc.tl.rank_genes_groups(adata, groupby="leiden_res0_3", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10, standard_scale='var', 
                                save = 'res_0_3_topmarkers_v1'
                               )

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden_res0_3", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10, standard_scale='var', 
                                save = 'res_0_3_topmarkers'
                               )

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden_res0_5", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10, standard_scale='var', 
                                save = 'res_0_5_topmarkers'
                               )

In [ ]:
sc.pl.rank_genes_groups_heatmap(adata, groupby="leiden_res0_3", n_genes = 20, show_gene_labels=True, cmap="RdBu_r", dendrogram = True,
                               vmin = 0, vmax = 4, vcenter = 2, swap_axes = True, save = 'leiden_res0_3.png')

In [ ]:
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key}": result[key][group]
     for group in groups for key in ['names', 'scores', 'pvals', 'pvals_adj', 'logfoldchanges']}
)
df.to_csv(f"{input_dir}/v1_res03.csv", index=False)

In [ ]:
#adata_hvg.var["mt"] = adata_hvg.var_names.str.startswith("MT-")
adata.var["mt"] = adata.var_names.str.startswith("MT-")

In [ ]:
#sc.pp.calculate_qc_metrics(adata_hvg, qc_vars=["mt"], inplace=True, log1p=True)
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, log1p=True)

In [ ]:
sc.pl.violin(
    adata,
    keys=["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    groupby="leiden_res0_3",
    stripplot=False,
    jitter=0,
    rotation=45,
    multi_panel=True,
    save = 'res03_disco.png'
)


In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)


In [ ]:
marker_dict = {
    "Adipocytic": [
        "ACACB", "GPAM", "MGST1", "PDE3B", "PPARG",
        "ACSL1", "GHR", "SOX5", "LINC00598", "PLIN1"
    ],
    "Cardiac": [
        "RYR2", "TTN", "FGF12", "MYBPC3", "SORBS2",
        "MLIP", "CTNNA3", "SLC8A1", "PALLD", "RBM20"
    ],
    "Endocardial": [
        "ECE1", "HSPG2", "PLPP3", "NFIA", "LEPR",
        "ST6GALNAC3", "ODF2L", "ITGA10", "PBX1", "SELP"
    ],
    "Endothelial": [
        "PLA2G5", "HSPG2", "ID3", "TIE1", "ST6GALNAC3",
        "NEXN", "IFI44L", "ADGRL4", "ARHGAP29", "PALMD"
    ],
    "Fibroblast": [
        "KAZN", "PLA2G5", "SPATA6", "ECHDC2", "NFIA",
        "NEGR1", "ST6GALNAC3", "TGFBR3", "DDR2", "PBX1"
    ],
    "Lymphoid": [
        "PIK3CD", "PLA2G5", "HSPG2", "CLIC4", "RUNX3",
        "EPB41", "LCK", "SMAP2", "NFIA", "JAK1"
    ],
    "Mesothelial": [
        "ERRFI1", "ENO1", "NPPA", "TNFRSF8", "PDPN",
        "PLA2G2A", "PLA2G5", "CLIC4", "MACF1", "MFSD2A"
    ],
    "Neuronal": [
        "PLA2G5", "CLIC4", "GRIK3", "MACF1", "TRABD2B",
        "SPATA6", "DOCK7", "ROR1", "DNAJC6", "NEGR1"
    ],
    "Perivascular": [
        "HES4", "VPS13D", "PLA2G5", "HSPG2", "HEYL",
        "ST6GALNAC3", "PALMD", "NGF", "RGS5", "AL499616.1"
    ]
}

In [ ]:
sc.pl.dotplot(
    adata,
    var_names = marker_dict,
    groupby   = "leiden_res0_3",   # change to your obs column
    dendrogram = False,
    save = 'disco_broad.png'
)

In [ ]:
marker_dict = {
    "CD14_monocytes": [
        "ZEB2", "RBM47", "SIPA1L1", "TBXAS1", "RBPJ",
        "VCAN", "PLXDC2", "MS4A6A", "SMAP2", "CD163",
        "DPYD", "FMN1", "TRPS1", "PTPRC", "AOAH"
    ],

    "CD16_monocytes": [
        "LYN", "PTPRC", "TCF7L2", "TBXAS1", "SAT1",
        "PRKCB", "RIPOR2", "AOAH", "ZEB2", "DOCK2",
        "MTSS1", "LYST", "SMAP2", "PTPRJ", "DOCK8"
    ],

    "Cycling_LYVE1_macrophages": [
        "F13A1", "RBPJ", "FMN1", "FRMD4B", "MRC1",
        "SLC9A9", "COLEC12", "RBM47", "CD163", "STAB1",
        "SYK", "MS4A6A", "ZEB2", "LRMDA", "ATP8B4"
    ],

    "LYVE1_macrophages": [
        "F13A1", "MRC1", "RBPJ", "FMN1", "CD163",
        "FRMD4B", "RBM47", "MAN1A1", "SLC9A9", "MS4A6A",
        "SLCO2B1", "COLEC12", "MERTK", "TBXAS1", "STAB1"
    ],

    "SPP1_macrophages": [
        "FMN1", "RBM47", "CPM", "CD163", "IQGAP2",
        "SLC1A3", "TBXAS1", "MSR1", "ATG7", "PLXDC2",
        "ZEB2", "MERTK", "MS4A6A", "RBPJ", "FRMD4B"
    ]
}

In [ ]:
sc.pl.dotplot(
    adata,
    var_names = marker_dict,
    groupby   = "leiden_res0_3",   # change to your obs column
    dendrogram = False,
    save = 'disco_fine.png'
)

In [ ]:
sc.pl.dotplot(
    adata, 
    var_names=['LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2',
              'C1QA', 'C1QC', 'CD68', 'CD163', 'HLA-DRA', 
              'SPP1', 'APOE',
              'FCGR2A', 'FCGR2B',
              'CLSPN', 'STIL'],
    groupby='leiden_res0_3',
    dot_max = 0.8,
    standard_scale='var', 
    color_map='RdBu_r' ,
    
 #  save = 'scvi_08.png'
)

In [ ]:
clusters = ["0", "1", "7"]
cluster_key = "leiden_res0_3"

adata_sub = adata[adata.obs[cluster_key].isin(clusters)].copy()
adata_sub.write('myeloid_v2.h5ad')

In [ ]:
clusters = ["4", "7"]
cluster_key = "leiden_res0_3"

adata_sub = adata[adata.obs[cluster_key].isin(clusters)].copy()


In [ ]:
clusters = ["4","5", "9"]         
cluster_key = "leiden_res0_5"            
# cells in those clusters (from adata_hvg)
sel_cells = adata_hvg.obs_names[adata_hvg.obs[cluster_key].astype(str).isin(clusters)]

# mask onto your full adata, then subset
adata.obs["mask"] = adata.obs_names.isin(sel_cells)
adata_subset = adata[adata.obs["mask"]].copy()
adata_subset.write('group3_myeloid.h5ad')

In [ ]:
marker_dict = {
    # ---- Myeloid focus ----
    "Classical Monocytes": ["S100A8", "S100A9", "VCAN", "LYZ", "LST1", "CTSS", "CD14"],
    "Nonclassical Monocytes": ["FCGR3A", "MS4A7", "LILRB1"],
    "Intermediate Monocytes": ["CD14", "FCGR3A", "MS4A7"],
    "Macrophages (TRM core)": ["C1QA", "C1QB", "C1QC", "APOE", "MRC1", "CD163", "MSR1", "CSF1R", "ITGAM"],
    "LYVE1+ TRM (cardiac)": ["LYVE1", "F13A1", "PDGFB"],
    "SPP1+ Macrophages": ["SPP1", "FABP5", "FN1"],
    "Cycling Myeloid": ["MKI67", "TOP2A"],
    "Dendritic Cells": ["HLA-DRA", "ITGAX"],
    "Neutrophils": ["S100A8", "S100A9"],

    # ---- Lymphoid (to exclude) ----
    "T Cells (CD4+)": ["CD3E", "IL7R"],
    "T Cells (CD8+)/NK": ["CD8A", "NKG7", "GNLY", "PRF1"],
    "B Cells": ["CD74", "MS4A7"],

    # ---- Cardiac parenchyma ----
    "Cardiomyocytes": ["TNNI3", "MYH7", "ACTC1", "MYL2"],
    "Endothelial Cells": ["PECAM1", "VWF", "KDR", "ESAM", "ENG"],
    "Fibroblasts": ["COL1A1", "COL1A2", "DCN", "LUM", "PDGFRA", "TAGLN"],
    "Smooth Muscle": ["ACTA2", "MYH11", "TAGLN"],
    "Pericytes": ["RGS5", "PDGFRB", "ABCC9"],
    "Epicardial/Mesothelial": ["WT1"],
    "Adipocytes (Cardiac/Pericardial)": ["FABP4"],
    "Erythroid": ["HBB", "HBA1"]
}


sc.pl.dotplot(
    adata_hvg, 
    var_names=marker_dict,
    groupby='leiden_res0_5',
    dot_max = 0.4,
    standard_scale='var', 
    color_map='RdBu_r' 
    
  # save = 'scvi_08.png'
)

In [ ]:
sc.pl.dotplot(
    adata_mac_hvg, 
    var_names=['LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2'],
    groupby='leiden_res0_8',
    dot_max = 0.8,
    standard_scale='var', 
    color_map='RdBu_r' ,
    
   save = 'scvi_08.png'
)\sc.pl.umap(
    adata_mac_hvg,
    color=['LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2', 
          'HLA-DQA1', 'HLA-DQB1', 'leiden_res0_8'],
    frameon=False,
    ncols=3, size = 30, legend_loc = 'on data', color_map = 'RdBu_r',
    save = 'tmp_umap'
)

In [ ]:
sc.pl.umap(
    adata_mac_hvg,
    color=['LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2', 
          'HLA-DQA1', 'HLA-DQB1', 'leiden_res0_8'],
    frameon=False,
    ncols=3, size = 30, legend_loc = 'on data', color_map = 'RdBu_r',
    save = 'tmp_umap'
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp

adata_obj = adata_mac_hvg          # your object
grp      = "leiden_res0_8"
targets  = {"0","3","5","11","14"}
combo    = "c0_3_5_11_14_vs_rest"
de_key   = "deg_combo"
gene_sets= "MSigDB_Hallmark_2020"

# 1) make two-level categorical label: target vs other
mask = adata_obj.obs[grp].astype(str).isin(targets)
adata_obj.obs[combo] = pd.Categorical(
    np.where(mask, "target", "other"),
    categories=["other", "target"]   # ensures both levels exist
)

# 2) DE: target vs other
sc.tl.rank_genes_groups(
    adata_obj, groupby=combo, groups=["target"], reference="other",
    method="wilcoxon", use_raw=False, key_added=de_key
)

# 3) build preranked list and run GSEA (tutorial style)
deg = sc.get.rank_genes_groups_df(adata_obj, group="target", key=de_key)
rnk = (deg[["names","logfoldchanges"]]
       .dropna().drop_duplicates("names")
       .set_index("names")["logfoldchanges"]
       .sort_values(ascending=False))

pre = gp.prerank(
    rnk=rnk, gene_sets=gene_sets, outdir=None,
    min_size=10, max_size=5000, permutation_num=1000,
    seed=42, threads=4          # use threads (processes is deprecated)
)

res = pre.res2d.copy()          # has columns like Term, NES, FDR q-val
res_up = res[res["NES"] > 0].sort_values("NES", ascending=False)

# (optional) quick plots
gp.barplot(res_up, column="NES", title=f"Upregulated terms: {combo}")
gp.dotplot(res_up, column="FDR q-val", title=f"Upregulated terms: {combo}")


In [ ]:
import scanpy as sc
import gseapy as gp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- inputs you set ----
adata_obj   = adata_mac_hvg          # your AnnData (HVG or all-genes)
group_var   = "leiden_res0_8"        # cluster column
target_lab  = "3"                    # e.g., cluster 3 (or "target" if you merged clusters)
de_key      = "deg_for_enrichr"      # where DE is stored
libraries   = ["MSigDB_Hallmark_2020", "GO_Biological_Process_2021", "Reactome_2016"]
organism    = "Human"                # "Mouse" if mouse symbols

# 1) DE: target vs rest (wilcoxon)
sc.tl.rank_genes_groups(
    adata_obj, groupby=group_var, groups=[target_lab], reference="rest",
    method="wilcoxon", use_raw=False, key_added=de_key
)

# 2) Build UP/DOWN gene lists (adjust thresholds as needed)
deg = sc.get.rank_genes_groups_df(adata_obj, group=target_lab, key=de_key).dropna()
padj_thr, lfc_thr, top_n = 0.05, 0.25, 200

deg_up = (deg.query("pvals_adj < @padj_thr and logfoldchanges >  @lfc_thr")
            .sort_values(["pvals_adj","logfoldchanges"], ascending=[True, False]))
deg_dn = (deg.query("pvals_adj < @padj_thr and logfoldchanges < -@lfc_thr")
            .sort_values(["pvals_adj","logfoldchanges"], ascending=[True, True]))

up_genes = deg_up["names"].drop_duplicates().head(top_n).tolist()
dn_genes = deg_dn["names"].drop_duplicates().head(top_n).tolist()

print(f"UP genes: {len(up_genes)}, DOWN genes: {len(dn_genes)}")

# 3) Enrichr ORA (GSEApy)
enr_up = gp.enrichr(
    gene_list=up_genes,
    gene_sets=libraries,
    organism=organism,
    outdir=None, # keep in-memory
    cutoff=1.0   # don't filter; we'll sort ourselves
)
enr_dn = gp.enrichr(
    gene_list=dn_genes,
    gene_sets=libraries,
    organism=organism,
    outdir=None,
    cutoff=1.0
)

# 4) Quick view: top terms per library (UP)
def show_top(enr_res, title, n=10):
    df = enr_res.res2d.copy()
    # unify column names a bit
    df.columns = [c.replace("_", " ") for c in df.columns]
    keep = df.rename(columns={
        "Adjusted P-value":"Adjusted P-value",
        "Combined Score":"Combined Score",
        "Term":"Term",
        "Gene_set":"Gene set"
    })
    print(f"\n{title}")
    for lib in keep["Gene set"].unique():
        sub = keep[keep["Gene set"]==lib].sort_values("Adjusted P-value").head(n)
        print(f"\n— {lib}")
        print(sub[["Term","Adjusted P-value","Combined Score"]].to_string(index=False))

show_top(enr_up, f"UP in {group_var}={target_lab}")
show_top(enr_dn, f"DOWN in {group_var}={target_lab}")

# 5) Tutorial-style plots
# Barplot and dotplot for UP (filter one library or stack all)
up_df = enr_up.res2d.copy()
lib_focus = "MSigDB_Hallmark_2020"   # change per plot
up_df = up_df[up_df["Gene_set"]==lib_focus].sort_values("Adjusted P-value").head(15)

gp.barplot(
    up_df,
    title=f"Enrichr (UP): {group_var}={target_lab} — {lib_focus}",
    column="Adjusted P-value"
)
gp.dotplot(
    up_df,
    title=f"Enrichr (UP): {group_var}={target_lab} — {lib_focus}",
    column="Adjusted P-value",
    cmap=None, size=6
)

# (Optional) same plots for DOWN
dn_df = enr_dn.res2d.copy()
dn_df = dn_df[dn_df["Gene_set"]==lib_focus].sort_values("Adjusted P-value").head(15)
gp.barplot(dn_df, title=f"Enrichr (DOWN): {group_var}={target_lab} — {lib_focus}",
           column="Adjusted P-value")
gp.dotplot(dn_df, title=f"Enrichr (DOWN): {group_var}={target_lab} — {lib_focus}",
           column="Adjusted P-value", cmap=None, size=6)


In [ ]:
adata_mac_hvg.obs

In [ ]:
obs_df = adata_mac_hvg.obs[['Disease', 'leiden_res0_8']]
frequency_df = obs_df.groupby(['Disease', 'leiden_res0_8']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(5, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters in obese / lean")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
plt.savefig("samplefreq.pdf", dpi=600)
plt.show()
